# 03. Сентимент-анализ
**Вход:** `articles_labeled_long.csv` (выход ноутбука 02)  
**Выход:**
- `sentiment_scores.csv` — все строки + `sentiment_score` (mxlcw)
- `sentiment_daily.csv` — дневная агрегация по тикеру

**Модель:** `mxlcw/rubert-tiny2-russian-financial-sentiment`  в блокноте с альтернативным прогоном использется `blanchefort/rubert-base-cased-sentiment-rusentiment`

**Score:** `p(positive) − p(negative)` ∈ [−1, 1]  
**Чанкинг:** до 4 чанков по 512 токенов (≈2048 токенов на статью)

In [ ]:
# ── ЯЧЕЙКА 1: Установка ──────────────────────────────────────
!pip install transformers torch -q

In [ ]:
# ── ЯЧЕЙКА 2: Настройки ──────────────────────────────────────
import os, re
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer
from tqdm.notebook import tqdm

INPUT_FILE       = '/content/articles_labeled_long.csv'
CHECKPOINT_FILE  = '/content/sentiment_checkpoint.csv'
OUTPUT_SCORES    = '/content/sentiment_scores.csv'
OUTPUT_DAILY     = '/content/sentiment_daily.csv'

MODEL_NAME  = 'mxlcw/rubert-tiny2-russian-financial-sentiment'
MAX_TOKENS  = 512
MAX_CHUNKS  = 4
BATCH_SAVE  = 500

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU"}')

classifier = pipeline(
    'text-classification',
    model=MODEL_NAME,
    return_all_scores=True,
    device=device,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✓ Модель загружена: {MODEL_NAME}')

In [ ]:
# ── ЯЧЕЙКА 3: Функции ─────────────────────────────────────────

def get_chunks(text, max_tokens=MAX_TOKENS, max_chunks=MAX_CHUNKS):
    """Разбиваем текст на чанки по max_tokens токенов."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, min(len(tokens), max_tokens * max_chunks), max_tokens):
        chunk_tokens = tokens[i : i + max_tokens]
        chunk_text   = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        if chunk_text.strip():
            chunks.append(chunk_text)
    return chunks if chunks else [text[:1000]]

def score_from_output(output):
    """p(positive) − p(negative) из списка {label, score}."""
    scores = {d['label'].lower(): d['score'] for d in output}
    return scores.get('positive', 0.0) - scores.get('negative', 0.0)

def sentiment_score(text):
    """Считаем score как среднее по чанкам."""
    if not isinstance(text, str) or len(text) < 50:
        return None
    chunks = get_chunks(text)
    scores = []
    for chunk in chunks:
        try:
            out = classifier(chunk, truncation=True, max_length=MAX_TOKENS)[0]
            scores.append(score_from_output(out))
        except Exception:
            continue
    return sum(scores) / len(scores) if scores else None

In [ ]:
# ── ЯЧЕЙКА 4: Основной прогон ─────────────────────────────────

df = pd.read_csv(INPUT_FILE, sep=';', encoding='utf-8-sig',
                 engine='c', lineterminator='\n')
df['text'] = df['text'].fillna('')
print(f'Строк для обработки: {len(df)}')

# Чекпоинт
already_done = set()
if os.path.exists(CHECKPOINT_FILE):
    ckpt = pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig')
    already_done = set(ckpt.index)
    print(f'✓ Чекпоинт: {len(ckpt)} записей')

results = []
to_process = df[~df.index.isin(already_done)]

for i, (idx, row) in enumerate(tqdm(to_process.iterrows(), total=len(to_process))):
    score = sentiment_score(row['text'])
    results.append({'original_idx': idx, 'sentiment_score': score})

    if (i + 1) % BATCH_SAVE == 0:
        batch = pd.DataFrame(results)
        if os.path.exists(CHECKPOINT_FILE):
            batch = pd.concat([pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig'), batch],
                              ignore_index=True)
        batch.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8-sig')
        results = []
        print(f'[{i+1}/{len(to_process)}] чекпоинт сохранён')

# Финал
scores_df = pd.DataFrame(results)
if os.path.exists(CHECKPOINT_FILE):
    scores_df = pd.concat([pd.read_csv(CHECKPOINT_FILE, encoding='utf-8-sig'), scores_df],
                          ignore_index=True)

df_out = df.reset_index().merge(scores_df, left_on='index', right_on='original_idx', how='left')
df_out.to_csv(OUTPUT_SCORES, index=False, encoding='utf-8-sig')

print(f'\n✓ Готово. Статистика sentiment_score:')
print(df_out['sentiment_score'].describe().to_string())

In [ ]:
# ── ЯЧЕЙКА 5: Дневная агрегация ───────────────────────────────

df_out = pd.read_csv(OUTPUT_SCORES, encoding='utf-8-sig')
df_out = df_out[df_out['sentiment_score'].notna()].copy()

# Нормализуем дату
df_out['date_parsed'] = pd.to_datetime(df_out['date'], format='mixed', dayfirst=True)
df_out['date_day']    = df_out['date_parsed'].dt.normalize()

daily = (
    df_out
    .groupby(['ticker', 'date_day'])['sentiment_score']
    .agg(sent_mean='mean', sent_count='count')
    .reset_index()
)

daily.to_csv(OUTPUT_DAILY, index=False, encoding='utf-8-sig')

print(f'Дневных наблюдений: {len(daily)}')
print(f'Тикеров: {daily["ticker"].nunique()}')
print(f'Период: {daily["date_day"].min()} — {daily["date_day"].max()}')
print(f'\n✓ Сохранено: {OUTPUT_DAILY}')